# Engram — Character Evaluation

Replicates **Section 4** of the FDG '26 paper: three polar-opposite personality profiles process the same six events and we compare the resulting memory representations.

**Target numbers (Table 1 from paper):**

| Metric | Paranoid Guard | Friendly Merchant | Rigid Clerk |
|--------|---------------|-------------------|-------------|
| Avg. Neuroticism score | 5.0 | 1.2 | 2.0 |
| Avg. Agreeableness score | 1.3 | 4.8 | 2.5 |
| Avg. Conscientiousness | 4.7 | 4.0 | 4.5 |
| Threat events (%) | 100 | 17 | 67 |
| Avg. importance | 9.7 | 7.8 | 7.8 |

In [ ]:
# Install dependencies (skip if already installed)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'google-genai', 'numpy', 'matplotlib', 'pandas', '-q'])

In [ ]:
import os, sys, json, time, copy
from pathlib import Path

# Add src/ to path so we can import the engram package
project_root = Path('..').resolve()
sys.path.insert(0, str(project_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({'figure.facecolor': '#FAFAFA', 'axes.facecolor': '#FAFAFA'})

print('Python path set. Project root:', project_root)

In [ ]:
# ── API key ──────────────────────────────────────────────────────────────────
# Set GEMINI_API_KEY in your environment or paste it below.
# In Colab: use userdata.get('GEMINI_API_KEY')

API_KEY = os.environ.get('GEMINI_API_KEY', '')

if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get('GEMINI_API_KEY')
    except Exception:
        pass

if not API_KEY:
    raise EnvironmentError('Set GEMINI_API_KEY before running this notebook.')

os.environ['GEMINI_API_KEY'] = API_KEY
print('API key loaded.')

In [ ]:
# ── Import Engram ─────────────────────────────────────────────────────────────
from engram.models import OCEANProfile, NPCConfig, Memory, EventTags
from engram.llm.client import GeminiClient
from engram.npc import NPCAgent

llm = GeminiClient(api_key=API_KEY)
print('Engram imported. GeminiClient ready.')

## 1. Personality Profiles (Paper § 4.1)

Three contrasting configurations processed through identical events.

In [ ]:
# ── NPC character shared across all three profiles ────────────────────────────
PERSONA = (
    "You are a townsperson in a small 17th-century English port town. "
    "You work at the dockside storeroom. You know the mayor, the local merchant, "
    "and most residents by name. You have lived here your whole life."
)

BACKSTORY = [
    "You grew up in this town and have worked the docks since you were twelve.",
    "The mayor has always seemed trustworthy and organized town affairs well.",
    "Strange things sometimes wash ashore — you have learned to be cautious.",
    "The harvest festival is the most important community event of the year.",
    "You keep the storeroom key on you at all times — it is your responsibility.",
    "Sickness travels fast in a port town; you have lost people to disease before.",
]

PROFILES = [
    # (npc_id, display_name, OCEAN vector)
    ('guard',    'Paranoid Guard',    OCEANProfile('Paranoid Guard',    O=0.2, C=0.5, E=0.3, A=0.2, N=0.9)),
    ('merchant', 'Friendly Merchant', OCEANProfile('Friendly Merchant', O=0.5, C=0.5, E=0.9, A=0.8, N=0.2)),
    ('clerk',    'Rigid Clerk',       OCEANProfile('Rigid Clerk',       O=0.1, C=0.9, E=0.3, A=0.5, N=0.4)),
]

# Pretty-print
for nid, label, p in PROFILES:
    print(f"{label:20s}  O={p.O}  C={p.C}  E={p.E}  A={p.A}  N={p.N}")

## 2. Event Sequences (Paper § 4.2)

Six events covering: physical threat, social opportunity, contradictory information, cooperative request, goal-relevant task, ambiguous novel stimulus.  
Each event is 2 dialogue turns (player message → NPC response × 2).

In [ ]:
EVENTS = [
    {
        'name': 'Physical Threat',
        'turns': [
            "Give me the storeroom key. Now. I know you have it.",
            "Don't make this difficult. Hand it over or things will get unpleasant.",
        ],
    },
    {
        'name': 'Social Opportunity',
        'turns': [
            "Hey! We're all getting together for the harvest dinner tonight. You should come!",
            "It'll be a great time — the whole town will be there. Music, food, dancing.",
        ],
    },
    {
        'name': 'Contradictory Information',
        'turns': [
            "I have evidence that Mayor Fletcher has been embezzling from the town fund for months.",
            "I've seen the ledgers myself. The numbers don't lie. What do you think we should do?",
        ],
    },
    {
        'name': 'Cooperative Request',
        'turns': [
            "Excuse me — my child is very sick and I can't afford a doctor. Could you help?",
            "Anything you can spare would mean the world. She has a fever and it's getting worse.",
        ],
    },
    {
        'name': 'Goal-Relevant Task',
        'turns': [
            "The morning shipment arrived two hours early. The dock master needs it processed before the market opens.",
            "There are twelve crates. Can you get started right away?",
        ],
    },
    {
        'name': 'Ambiguous Novel Stimulus',
        'turns': [
            "There's a strange animal loose in the town square. Nobody knows what it is or where it came from.",
            "It hasn't attacked anyone but people are frightened. Some say it's a bad omen.",
        ],
    },
]

print(f'{len(EVENTS)} events × 2 turns = {len(EVENTS)*2} total turns per NPC')
print(f'3 NPCs × {len(EVENTS)*2} turns = {3 * len(EVENTS) * 2} total LLM pipeline calls')

## 3. Run the Evaluation

Each NPC processes all six events. We capture:
- Whether threat was detected per turn
- All memories stored (with their tags)
- Importance scores and OCEAN affinity tags

In [ ]:
# ── Monkey-patch assess_threat to capture is_threat per turn ─────────────────
# We intercept the ThreatAssessment before it flows to the agent so we can log it.

import engram.pipeline.threat as _threat_module
_original_assess = _threat_module.assess_threat

_threat_log = {}  # npc_id -> list[bool]

def _patched_assess(player_input, profile, memory_manager, llm_client):
    result = _original_assess(player_input, profile, memory_manager, llm_client)
    nid = memory_manager.npc_id
    _threat_log.setdefault(nid, []).append(result.is_threat)
    return result

_threat_module.assess_threat = _patched_assess
print('Threat logger patched.')

In [ ]:
# ── Create NPCAgents (fresh data dir per eval run) ────────────────────────────
import shutil

EVAL_DATA_DIR = str(project_root / 'eval' / 'eval_data')
# Wipe previous eval data to start clean
shutil.rmtree(EVAL_DATA_DIR, ignore_errors=True)
os.makedirs(EVAL_DATA_DIR, exist_ok=True)

agents = {}
for npc_id, label, profile in PROFILES:
    config = NPCConfig(
        npc_id=npc_id,
        name=label,
        persona=PERSONA,
        backstory=BACKSTORY,
        profile=profile,
        initial_facts=[],
    )
    print(f'Initializing {label}... (embedding backstory)', end=' ', flush=True)
    agent = NPCAgent(config, llm, data_dir=EVAL_DATA_DIR)
    agents[npc_id] = (label, agent)
    print(f'done. {len(agent.memory_manager.all_memories)} backstory memories loaded.')

In [ ]:
# ── Run all events ────────────────────────────────────────────────────────────
import textwrap

def wrap(text, width=55):
    return '\n'.join(textwrap.wrap(str(text), width))

responses_log = {}  # npc_id -> list[dict]

for event in EVENTS:
    print(f"\n{'='*70}")
    print(f"EVENT: {event['name']}")
    print('='*70)
    
    for turn_text in event['turns']:
        print(f"\n  Player: {turn_text[:80]}")
        for npc_id, (label, agent) in agents.items():
            mems_before = len(agent.memory_manager.all_memories)
            response = agent.run_turn(turn_text)
            mems_after = len(agent.memory_manager.all_memories)
            stored = mems_after > mems_before
            
            responses_log.setdefault(npc_id, []).append({
                'event': event['name'],
                'turn': turn_text,
                'response': response,
                'stored': stored,
            })
            
            print(f"  [{label[:16]:16s}] {wrap(response[:100])} {'[+mem]' if stored else ''}")
        
        time.sleep(1)  # rate limiting

print('\n✅ All events complete.')

In [ ]:
# ── End sessions ─────────────────────────────────────────────────────────────
for npc_id, (label, agent) in agents.items():
    print(f'Ending session for {label}...', end=' ', flush=True)
    agent.end_session()
    print('done.')

## 4. Compute Metrics

From stored memories: average OCEAN affinity tags (N, A, C), threat rate, average importance.

In [ ]:
def compute_metrics(agent, npc_id, label):
    mems = agent.memory_manager.all_memories
    # Exclude backstory (pre-loaded) — focus on interaction memories
    interaction_mems = [m for m in mems if m.source in ('session', 'interaction')]
    all_mems = mems  # use all for OCEAN tag averages
    
    if not all_mems:
        return {}
    
    # Average OCEAN tags from memories (1–5 scale)
    avg_N = np.mean([m.tags.ocean.get('N', 3) for m in all_mems])
    avg_A = np.mean([m.tags.ocean.get('A', 3) for m in all_mems])
    avg_C = np.mean([m.tags.ocean.get('C', 3) for m in all_mems])
    avg_O = np.mean([m.tags.ocean.get('O', 3) for m in all_mems])
    avg_E = np.mean([m.tags.ocean.get('E', 3) for m in all_mems])
    
    # Average importance
    avg_imp = np.mean([m.tags.importance for m in all_mems])
    
    # Threat rate from logged assessments
    threat_flags = _threat_log.get(npc_id, [])
    threat_pct = (sum(threat_flags) / len(threat_flags) * 100) if threat_flags else 0
    
    # Average threat_level tag on stored memories
    avg_threat_tag = np.mean([m.tags.threat_level for m in all_mems])
    
    # Memory strength distribution
    strengths = [m.score for m in all_mems]
    
    # Per-event storage count
    stored_per_event = {}
    for entry in responses_log.get(npc_id, []):
        e = entry['event']
        stored_per_event.setdefault(e, 0)
        if entry['stored']:
            stored_per_event[e] += 1
    
    return {
        'label': label,
        'npc_id': npc_id,
        'n_memories': len(all_mems),
        'n_interaction': len(interaction_mems),
        'avg_N': round(avg_N, 2),
        'avg_A': round(avg_A, 2),
        'avg_C': round(avg_C, 2),
        'avg_O': round(avg_O, 2),
        'avg_E': round(avg_E, 2),
        'avg_importance': round(avg_imp, 2),
        'threat_pct': round(threat_pct, 1),
        'avg_threat_tag': round(avg_threat_tag, 3),
        'strengths': strengths,
        'stored_per_event': stored_per_event,
        'n_threats': sum(threat_flags),
        'n_turns': len(threat_flags),
    }

results = []
for npc_id, (label, agent) in agents.items():
    m = compute_metrics(agent, npc_id, label)
    results.append(m)
    print(f"{label}: {m['n_memories']} memories, threat={m['threat_pct']}%, avg_N={m['avg_N']}, avg_imp={m['avg_importance']}")

## 5. Table 1 — Comparison with Paper

In [ ]:
# ── Our results ──────────────────────────────────────────────────────────────
our_rows = [
    ['Avg. Neuroticism score'] + [r['avg_N'] for r in results],
    ['Avg. Agreeableness score'] + [r['avg_A'] for r in results],
    ['Avg. Conscientiousness'] + [r['avg_C'] for r in results],
    ['Threat events (%)'] + [r['threat_pct'] for r in results],
    ['Avg. importance'] + [r['avg_importance'] for r in results],
]

cols = ['Metric'] + [r['label'] for r in results]
df_ours = pd.DataFrame(our_rows, columns=cols)

# ── Paper's reported numbers (Table 1) ───────────────────────────────────────
paper_rows = [
    ['Avg. Neuroticism score',  5.0, 1.2, 2.0],
    ['Avg. Agreeableness score', 1.3, 4.8, 2.5],
    ['Avg. Conscientiousness',  4.7, 4.0, 4.5],
    ['Threat events (%)',       100, 17,  67],
    ['Avg. importance',         9.7, 7.8, 7.8],
]
df_paper = pd.DataFrame(paper_rows, columns=['Metric', 'Paranoid Guard', 'Friendly Merchant', 'Rigid Clerk'])

print('\n── OUR RESULTS ──────────────────────────────────────────────────────')
print(df_ours.to_string(index=False))

print('\n── PAPER TABLE 1 (target) ────────────────────────────────────────────')
print(df_paper.to_string(index=False))

In [ ]:
# ── Delta vs paper ────────────────────────────────────────────────────────────
# Align by metric name; compute absolute difference
paper_vals = {
    'guard':    {'avg_N': 5.0, 'avg_A': 1.3, 'avg_C': 4.7, 'threat_pct': 100, 'avg_importance': 9.7},
    'merchant': {'avg_N': 1.2, 'avg_A': 4.8, 'avg_C': 4.0, 'threat_pct': 17,  'avg_importance': 7.8},
    'clerk':    {'avg_N': 2.0, 'avg_A': 2.5, 'avg_C': 4.5, 'threat_pct': 67,  'avg_importance': 7.8},
}

print('\n── DELTA vs PAPER (|ours - paper|) ──────────────────────────────────')
metric_keys = [('avg_N','Avg. N'), ('avg_A','Avg. A'), ('avg_C','Avg. C'),
               ('threat_pct','Threat %'), ('avg_importance','Avg. Imp')]
delta_rows = []
for key, label in metric_keys:
    row = [label]
    for r in results:
        nid = r['npc_id']
        delta = abs(r[key] - paper_vals[nid][key])
        row.append(round(delta, 2))
    delta_rows.append(row)

df_delta = pd.DataFrame(delta_rows, columns=['Metric'] + [r['label'] for r in results])
print(df_delta.to_string(index=False))

## 6. Visualizations

In [ ]:
# ── Fig 1: OCEAN tag averages per NPC (grouped bar chart) ────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {'guard': '#DC2626', 'merchant': '#0D9488', 'clerk': '#7C3AED'}
trait_keys = [('avg_O','O'), ('avg_C','C'), ('avg_E','E'), ('avg_A','A'), ('avg_N','N')]

for ax, r in zip(axes, results):
    traits = [label for _, label in trait_keys]
    ours   = [r[key] for key, _ in trait_keys]
    paper  = [paper_vals.get(r['npc_id'], {}).get(key.replace('avg_', 'avg_'), None)
              for key, _ in trait_keys]
    
    x = np.arange(len(traits))
    ax.bar(x - 0.2, ours, 0.35, label='Ours', color=colors[r['npc_id']], alpha=0.85)
    # Paper only has N, A, C — mark those
    paper_N = paper_vals[r['npc_id']]['avg_N']
    paper_A = paper_vals[r['npc_id']]['avg_A']
    paper_C = paper_vals[r['npc_id']]['avg_C']
    ax.axhline(paper_N, xmin=0.78, xmax=0.98, color='black', linestyle='--', linewidth=1.2, alpha=0.6)
    ax.axhline(paper_A, xmin=0.58, xmax=0.78, color='black', linestyle='--', linewidth=1.2, alpha=0.6)
    ax.axhline(paper_C, xmin=0.18, xmax=0.38, color='black', linestyle='--', linewidth=1.2, alpha=0.6)
    ax.text(4.3, paper_N, f'paper: {paper_N}', fontsize=8, va='center', color='black', alpha=0.6)
    ax.text(3.3, paper_A, f'paper: {paper_A}', fontsize=8, va='center', color='black', alpha=0.6)
    ax.text(1.3, paper_C, f'paper: {paper_C}', fontsize=8, va='center', color='black', alpha=0.6)
    
    ax.set_xticks(x)
    ax.set_xticklabels(traits)
    ax.set_ylim(0, 5.5)
    ax.set_ylabel('Avg OCEAN tag (1–5)')
    ax.set_title(r['label'], fontweight='bold', color=colors[r['npc_id']])
    ax.grid(axis='y', alpha=0.2)

plt.suptitle('Avg Memory OCEAN Tags vs Paper (dashes = paper targets for N, A, C)', fontsize=12)
plt.tight_layout()
plt.savefig('ocean_tags.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ocean_tags.png')

In [ ]:
# ── Fig 2: Threat classification rate comparison ──────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

labels = [r['label'] for r in results]
our_threat = [r['threat_pct'] for r in results]
paper_threat = [100, 17, 67]  # Guard, Merchant, Clerk

x = np.arange(len(labels))
w = 0.35
ax.bar(x - w/2, our_threat, w, label='Ours', color=['#DC2626', '#0D9488', '#7C3AED'], alpha=0.85)
ax.bar(x + w/2, paper_threat, w, label='Paper', color='#94A3B8', alpha=0.7, edgecolor='black', linewidth=0.8)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Threat Events (%)')
ax.set_ylim(0, 115)
ax.legend()
ax.set_title('Threat Classification Rate: Ours vs Paper', fontweight='bold')
ax.grid(axis='y', alpha=0.2)

for i, (ours, paper) in enumerate(zip(our_threat, paper_threat)):
    ax.text(i - w/2, ours + 1, f'{ours:.0f}%', ha='center', fontsize=10, fontweight='bold')
    ax.text(i + w/2, paper + 1, f'{paper}%', ha='center', fontsize=10, color='#475569')

plt.tight_layout()
plt.savefig('threat_rate.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: threat_rate.png')

In [ ]:
# ── Fig 3: Importance distribution per NPC ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

for r in results:
    imps = [m.tags.importance for m in agents[r['npc_id']][1].memory_manager.all_memories]
    ax.hist(imps, bins=range(1, 12), alpha=0.65, label=r['label'],
            color=colors[r['npc_id']], edgecolor='white', density=True)

ax.set_xlabel('Importance score (1–10)')
ax.set_ylabel('Density')
ax.set_title('Memory Importance Distribution by Personality', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.2)

# Mark paper averages
for r, paper_avg in zip(results, [9.7, 7.8, 7.8]):
    ax.axvline(paper_avg, color=colors[r['npc_id']], linestyle=':', linewidth=1.5, alpha=0.8)

plt.tight_layout()
plt.savefig('importance_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: importance_dist.png')

In [ ]:
# ── Fig 4: Radar chart — OCEAN tag profile per NPC ───────────────────────────
from matplotlib.patches import FancyArrowPatch

categories = ['O', 'C', 'E', 'A', 'N']
N_cat = len(categories)
angles = [n / float(N_cat) * 2 * np.pi for n in range(N_cat)]
angles += angles[:1]

fig, ax = plt.subplots(1, 1, figsize=(6, 6), subplot_kw=dict(polar=True))

for r in results:
    vals = [r[f'avg_{t}'] for t in categories]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, label=r['label'], color=colors[r['npc_id']])
    ax.fill(angles, vals, alpha=0.1, color=colors[r['npc_id']])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=13)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1', '2', '3', '4', '5'], fontsize=8)
ax.set_title('Memory OCEAN Tag Profiles\n(avg per-memory affinity, 1–5)', fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig('radar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: radar.png')

In [ ]:
# ── Fig 5: Memory count and threat flag per event ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

event_names = [e['name'] for e in EVENTS]
x = np.arange(len(event_names))
w = 0.25

# Left: memories stored per event
ax = axes[0]
for i, r in enumerate(results):
    stored = [r['stored_per_event'].get(e['name'], 0) for e in EVENTS]
    ax.bar(x + (i - 1) * w, stored, w, label=r['label'], color=colors[r['npc_id']], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(event_names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Turns where memory was stored')
ax.set_title('Memory Storage per Event', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.2)

# Right: average threat_level tag per NPC + paper's threat %
ax = axes[1]
npc_labels = [r['label'] for r in results]
avg_threat_tags = [r['avg_threat_tag'] for r in results]
our_threat_pct_norm = [r['threat_pct'] / 100 for r in results]

ax.bar(x - 0.2, avg_threat_tags, 0.35, label='Avg threat_level tag (0–1)', 
       color=[colors[r['npc_id']] for r in results], alpha=0.85)
ax.bar(x + 0.2, our_threat_pct_norm, 0.35, label='Threat flag rate (normalized)', 
       color=[colors[r['npc_id']] for r in results], alpha=0.4, edgecolor='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(npc_labels, fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score (0–1)')
ax.set_title('Threat Sensitivity: Tag vs Flag Rate', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig('event_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: event_analysis.png')

## 7. Final Results Table

In [ ]:
# ── Full results dump ─────────────────────────────────────────────────────────
print('=' * 72)
print('ENGRAM CHARACTER EVAL — FULL RESULTS')
print('=' * 72)

for r in results:
    agent = agents[r['npc_id']][1]
    eff = agent.profile.effective
    print(f"\n{r['label']} ({r['npc_id']})")
    print(f"  Baseline OCEAN  : O={agent.config.profile.O}  C={agent.config.profile.C}  "
          f"E={agent.config.profile.E}  A={agent.config.profile.A}  N={agent.config.profile.N}")
    print(f"  Effective (post-decay): " + '  '.join(f"{k}={v:.3f}" for k, v in eff.items()))
    print(f"  Total memories  : {r['n_memories']} (backstory + interaction)")
    print(f"  Avg N-tag       : {r['avg_N']}   (paper: {paper_vals[r['npc_id']]['avg_N']})")
    print(f"  Avg A-tag       : {r['avg_A']}   (paper: {paper_vals[r['npc_id']]['avg_A']})")
    print(f"  Avg C-tag       : {r['avg_C']}   (paper: {paper_vals[r['npc_id']]['avg_C']})")
    print(f"  Threat rate     : {r['threat_pct']}%  (paper: {paper_vals[r['npc_id']]['threat_pct']}%)")
    print(f"  Avg importance  : {r['avg_importance']}   (paper: {paper_vals[r['npc_id']]['avg_importance']})")

print('\n' + '=' * 72)
print('Trend check (higher is more correct):')
n_vals = {r['npc_id']: r['avg_N'] for r in results}
a_vals = {r['npc_id']: r['avg_A'] for r in results}
print(f"  Guard N > Clerk N > Merchant N : {n_vals['guard']} > {n_vals['clerk']} > {n_vals['merchant']} → "
      f"{'✅' if n_vals['guard'] > n_vals['clerk'] > n_vals['merchant'] else '❌'}")
print(f"  Merchant A > Clerk A > Guard A : {a_vals['merchant']} > {a_vals['clerk']} > {a_vals['guard']} → "
      f"{'✅' if a_vals['merchant'] > a_vals['clerk'] > a_vals['guard'] else '❌'}")
threat_vals = {r['npc_id']: r['threat_pct'] for r in results}
print(f"  Guard threat > Clerk threat > Merchant threat : "
      f"{threat_vals['guard']} > {threat_vals['clerk']} > {threat_vals['merchant']} → "
      f"{'✅' if threat_vals['guard'] > threat_vals['clerk'] > threat_vals['merchant'] else '❌'}")

In [ ]:
# ── Save all results to JSON for record-keeping ───────────────────────────────
save_results = []
for r in results:
    rec = {k: v for k, v in r.items() if k != 'strengths'}  # skip raw list
    save_results.append(rec)

out_path = project_root / 'eval' / 'results.json'
with open(out_path, 'w') as f:
    json.dump({
        'paper_table1': {
            'guard':    paper_vals['guard'],
            'merchant': paper_vals['merchant'],
            'clerk':    paper_vals['clerk'],
        },
        'our_results': save_results,
    }, f, indent=2)

print(f'Results saved to {out_path}')

## Interpretation

The key claim in the paper is **ordinal correctness** — not exact number matching:

| Ordering check | What it means |
|---|---|
| `Guard N > Clerk N > Merchant N` | High-N NPCs attract threat-coded memories more |
| `Merchant A > Clerk A > Guard A` | High-A NPCs attract socially warm memories more |
| `Guard threat% > Clerk threat% > Merchant threat%` | High-N NPCs classify more events as threats |

If those orderings hold, the personality-parameterized encoding is working as intended. Absolute values will differ from the paper because:
- The paper ran 13–14 total turns (we ran 12)
- Gemini's tagging has LLM variance
- Paper backstory may have been more richly authored